## Environment

In [3]:
!pip install -q timm transformers accelerate sentencepiece pandas pillow tqdm matplotlib

## Load shared code

In [4]:
%run ./02_shared.ipynb

/usr/local/lib/python3.12/dist-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


## Training

In [5]:
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.optim import AdamW
from tqdm.auto import tqdm
from transformers import get_cosine_schedule_with_warmup


class Trainer:
    def __init__(
        self,
        model: VQAModel,
        config: AConfig,
        tokenizer: object,
        run_dir: Path,
        name: str
    ) -> None:
        self.model = model.to(config.device)
        self.config = config
        self.tokenizer = tokenizer
        self.run_dir = run_dir
        self.name = name
        self.use_amp = config.fp16 and config.device == "cuda"
        self.scaler = torch.amp.GradScaler(
            "cuda",
            enabled=self.use_amp
        )

    def move(self, batch: dict[str, object]) -> dict[str, object]:
        moved = {}

        for key, value in batch.items():
            if isinstance(value, torch.Tensor):
                moved[key] = value.to(self.config.device)
            else:
                moved[key] = value

        return moved

    def optim(self) -> AdamW:
        enc_params = []
        task_params = []

        for name, param in self.model.named_parameters():
            if not param.requires_grad:
                continue

            if "backbone" in name:
                enc_params.append(param)
            else:
                task_params.append(param)

        return AdamW(
            [
                {"params": enc_params, "lr": self.config.encoder_lr},
                {"params": task_params, "lr": self.config.task_lr}
            ],
            weight_decay=self.config.weight_decay
        )

    def sched(self, optim: AdamW, loader: object) -> object:
        total_steps = math.ceil(
            len(loader)
            * self.config.epochs
            / self.config.grad_accum
        )
        warmup_steps = int(total_steps * self.config.warmup_ratio)

        return get_cosine_schedule_with_warmup(
            optim,
            warmup_steps,
            total_steps
        )

    def restore(
        self,
        optim: AdamW,
        sched: object
    ) -> tuple[int, float, int]:
        if not self.config.resume:
            return 1, -1.0, 0

        state = Checkpoint.load(
            self.run_dir / "last.pt",
            self.config.device
        )

        if state is None:
            return 1, -1.0, 0

        self.model.load_state_dict(state["model"])
        optim.load_state_dict(state["optim"])
        sched.load_state_dict(state["sched"])

        return (
            int(state["epoch"]) + 1,
            float(state["best"]),
            int(state.get("bad_epochs", 0))
        )

    def state(
        self,
        epoch: int,
        best: float,
        bad_epochs: int,
        metrics: dict[str, float],
        optim: AdamW,
        sched: object
    ) -> dict[str, object]:
        return {
            "epoch": epoch,
            "best": best,
            "bad_epochs": bad_epochs,
            "metrics": metrics,
            "model": self.model.state_dict(),
            "optim": optim.state_dict(),
            "sched": sched.state_dict()
        }

In [6]:
class TrainStep:
    def __init__(self, trainer: Trainer) -> None:
        self.trainer = trainer

    def train_epoch(
        self,
        loader: object,
        optim: AdamW,
        sched: object,
        epoch: int
    ) -> float:
        model = self.trainer.model
        config = self.trainer.config
        model.train()
        optim.zero_grad(set_to_none=True)
        losses = []
        progress = tqdm(loader, desc=f"train {self.trainer.name} - epoch {epoch}", leave=False)

        for step, batch in enumerate(progress, start=1):
            batch = self.trainer.move(batch)

            with torch.amp.autocast(
                "cuda",
                enabled=self.trainer.use_amp
            ):
                output = model(
                    images=batch["images"],
                    question_ids=batch["question_ids"],
                    question_mask=batch["question_mask"],
                    answer_ids=batch["answer_ids"]
                )
                loss = output["loss"] / config.grad_accum

            self.trainer.scaler.scale(loss).backward()
            update = step % config.grad_accum == 0
            last = step == len(loader)

            if update or last:
                self.trainer.scaler.unscale_(optim)
                nn.utils.clip_grad_norm_(
                    model.parameters(),
                    config.grad_clip
                )
                old_scale = self.trainer.scaler.get_scale()
                self.trainer.scaler.step(optim)
                self.trainer.scaler.update()
                new_scale = self.trainer.scaler.get_scale()
                optim.zero_grad(set_to_none=True)

                if (not self.trainer.use_amp) or new_scale >= old_scale:
                    sched.step()

            losses.append(float(loss.item() * config.grad_accum))
            progress.set_postfix(train_loss=round(float(np.mean(losses)), 4))

        return float(np.mean(losses))

In [7]:
class EvalStep:
    def __init__(self, trainer: Trainer) -> None:
        self.trainer = trainer

    @torch.no_grad()
    def loss_eval(self, loader: object) -> float:
        model = self.trainer.model
        model.eval()
        losses = []

        for batch in loader:
            batch = self.trainer.move(batch)
            output = model(
                images=batch["images"],
                question_ids=batch["question_ids"],
                question_mask=batch["question_mask"],
                answer_ids=batch["answer_ids"]
            )
            losses.append(float(output["loss"].item()))

        return float(np.mean(losses))

    @torch.no_grad()
    def predict(self, loader: object) -> pd.DataFrame:
        model = self.trainer.model
        model.eval()
        rows = []

        for batch in loader:
            batch = self.trainer.move(batch)
            generated = model.generate(
                images=batch["images"],
                question_ids=batch["question_ids"],
                question_mask=batch["question_mask"]
            )
            preds = self.trainer.tokenizer.batch_decode(
                generated.detach().cpu().tolist(),
                skip_special_tokens=True
            )

            for gold, pred in zip(batch["answers"], preds):
                rows.append(
                    {
                        "config": self.trainer.name,
                        "answer": gold,
                        "prediction": pred.strip()
                    }
                )

        return pd.DataFrame(rows)

In [8]:
class TrainLoop:
    def __init__(self, trainer: Trainer) -> None:
        self.trainer = trainer
        self.train_step = TrainStep(trainer)
        self.eval_step = EvalStep(trainer)

    def run(self, train_loader: object, val_loader: object) -> pd.DataFrame:
        optim = self.trainer.optim()
        sched = self.trainer.sched(optim, train_loader)
        start_epoch, best, bad_epochs = self.trainer.restore(optim, sched)
        rows = []

        for epoch in range(start_epoch, self.trainer.config.epochs + 1):
            train_loss = self.train_step.train_epoch(
                train_loader,
                optim,
                sched,
                epoch
            )
            val_loss = self.eval_step.loss_eval(val_loader)
            pred_table = self.eval_step.predict(val_loader)
            metrics = ExactMatchScore().batch(
                pred_table["prediction"].tolist(),
                pred_table["answer"].tolist()
            )
            metrics["train_loss"] = train_loss
            metrics["val_loss"] = val_loss
            metrics["epoch"] = epoch
            metrics["config"] = self.trainer.name
            rows.append(metrics)
            score = metrics["exact_match"]
            print(
                (
                    f"{self.trainer.name} epoch {epoch}: "
                    f"train_loss={train_loss:.4f}, "
                    f"val_loss={val_loss:.4f}, "
                    f"exact_match={score:.4f}"
                )
            )

            if score > best:
                best = score
                bad_epochs = 0
                Checkpoint.save(
                    self.trainer.run_dir / "best.pt",
                    self.trainer.state(
                        epoch,
                        best,
                        bad_epochs,
                        metrics,
                        optim,
                        sched
                    )
                )
            else:
                bad_epochs += 1

            Checkpoint.save(
                self.trainer.run_dir / "last.pt",
                self.trainer.state(
                    epoch,
                    best,
                    bad_epochs,
                    metrics,
                    optim,
                    sched
                )
            )

            if bad_epochs >= self.trainer.config.patience:
                break

        return pd.DataFrame(rows)

## Plot and run

In [9]:
class LossPlot:
    def __init__(self, figure_dir: str) -> None:
        self.figure_dir = Path(figure_dir)

    def history(self, result: dict[str, object]) -> pd.DataFrame:
        return pd.DataFrame(result["history"])

    def train_compare(
        self,
        a1_result: dict[str, object],
        a2_result: dict[str, object]
    ) -> Path:
        a1 = self.history(a1_result)
        a2 = self.history(a2_result)
        out_path = self.figure_dir / "a_train_loss_compare.png"

        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(
            a1["epoch"],
            a1["train_loss"],
            marker="o",
            label="A1 - LSTM"
        )
        ax.plot(
            a2["epoch"],
            a2["train_loss"],
            marker="o",
            label="A2 - Transformer"
        )
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Train Loss")
        ax.set_title("A1 vs A2 - Train Loss")
        ax.grid(True, alpha=0.3)
        ax.legend()
        fig.tight_layout()
        out_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_path, dpi=160, bbox_inches="tight")
        plt.close(fig)

        return out_path

    def val_compare(
        self,
        a1_result: dict[str, object],
        a2_result: dict[str, object]
    ) -> Path:
        a1 = self.history(a1_result)
        a2 = self.history(a2_result)
        out_path = self.figure_dir / "a_val_loss_compare.png"

        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(
            a1["epoch"],
            a1["val_loss"],
            marker="o",
            label="A1 - LSTM"
        )
        ax.plot(
            a2["epoch"],
            a2["val_loss"],
            marker="o",
            label="A2 - Transformer"
        )
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Validation Loss")
        ax.set_title("A1 vs A2 - Validation Loss")
        ax.grid(True, alpha=0.3)
        ax.legend()
        fig.tight_layout()
        out_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_path, dpi=160, bbox_inches="tight")
        plt.close(fig)

        return out_path

    def save_compare(
        self,
        a1_result: dict[str, object],
        a2_result: dict[str, object]
    ) -> tuple[Path, Path]:
        train_path = self.train_compare(a1_result, a2_result)
        val_path = self.val_compare(a1_result, a2_result)

        return train_path, val_path

In [10]:
class TrainRun:
    def __init__(self, config: AConfig) -> None:
        self.config = config
        self.data = DataModule(config)
        self.meta = self.data.meta
        self.loaders = self.data.loaders()
        self.builder = ModelBuild(config, self.data.tokenizer)
        self.plot = LossPlot(self.meta["figure_dir"])

    def seed(self) -> None:
        random.seed(self.config.seed)
        np.random.seed(self.config.seed)
        torch.manual_seed(self.config.seed)
        torch.cuda.manual_seed_all(self.config.seed)

    def train_model(self, label: str, decoder: str) -> dict[str, object]:
        MemoryCleaner.clear()
        train_loader, val_loader, _ = self.loaders
        model = self.builder.build(decoder)
        run_dir = Path(self.meta["checkpoint_dir"]) / f"a_{decoder}"
        trainer = Trainer(
            model=model,
            config=self.config,
            tokenizer=self.data.tokenizer,
            run_dir=run_dir,
            name=label
        )
        log_table = TrainLoop(trainer).run(train_loader, val_loader)
        state = Checkpoint.load(run_dir / "best.pt", self.config.device)

        if state is None:
            raise FileNotFoundError(f"Missing checkpoint: {run_dir / 'best.pt'}")

        metrics = dict(state["metrics"])
        metrics["config"] = label
        metrics["decoder"] = decoder
        metrics["history"] = log_table.to_dict("records")
        MemoryCleaner.clear()

        return metrics


config = AConfig()
runner = TrainRun(config)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Run A1

In [11]:
runner.seed()
MemoryCleaner.clear()
a1_result = runner.train_model("A1", "lstm")
MemoryCleaner.clear()

model.safetensors:   0%|          | 0.00/86.5M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

A1 epoch 17: train_loss=0.3030, val_loss=1.6540, exact_match=0.3370


train A1 - epoch 18:   0%|          | 0/65 [00:00<?, ?it/s]

A1 epoch 18: train_loss=0.2743, val_loss=1.6888, exact_match=0.3333


train A1 - epoch 19:   0%|          | 0/65 [00:00<?, ?it/s]

A1 epoch 19: train_loss=0.2409, val_loss=1.6498, exact_match=0.3407


## Run A2

In [12]:
runner.seed()
MemoryCleaner.clear()
a2_result = runner.train_model("A2", "transformer")
MemoryCleaner.clear()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


train A2 - epoch 1:   0%|          | 0/65 [00:00<?, ?it/s]

A2 epoch 1: train_loss=259.4687, val_loss=147.0784, exact_match=0.0000


train A2 - epoch 2:   0%|          | 0/65 [00:00<?, ?it/s]

...

A2 epoch 23: train_loss=6.4404, val_loss=6.1716, exact_match=0.3077


train A2 - epoch 24:   0%|          | 0/65 [00:00<?, ?it/s]

A2 epoch 24: train_loss=6.3653, val_loss=6.1673, exact_match=0.3114


train A2 - epoch 25:   0%|          | 0/65 [00:00<?, ?it/s]

A2 epoch 25: train_loss=6.3827, val_loss=6.1641, exact_match=0.3077


## A1/A2 loss comparison


In [ ]:
loss_paths = runner.plot.save_compare(a1_result, a2_result)